<a href="https://colab.research.google.com/github/Janani-12k/ML/blob/main/Candidate_Elimination_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
class CandidateElimination:
    def __init__(self, num_features):
        self.num_features = num_features
        # Initialize S with the most specific hypothesis ('0' represents 'no value')
        self.S = [('0',) * num_features]
        # Initialize G with the most general hypothesis ('?' represents 'any value')
        self.G = [('?',) * num_features]

    def fit(self, X, y):
        for i, (x, label) in enumerate(zip(X, y)):
            print(f"\nTraining example {i+1}: {x} with label {label}")
            if label == 1:  # Positive example
                self.S = self._generalize_S(x)
                self.G = self._remove_inconsistent_G(x)
            else:  # Negative example
                self.S = self._remove_inconsistent_S(x)
                self.G = self._specialize_G(x)

            # Prune hypotheses to maintain a minimal set
            self._prune_hypotheses()

            print(f"S after update: {self.S}")
            print(f"G after update: {self.G}")

    def _is_consistent(self, hypothesis, example):
        """Checks if a hypothesis is consistent with an example."""
        return all(h == '?' or h == e for h, e in zip(hypothesis, example))

    def _generalize_S(self, positive_example):
        S_new = []
        for s in self.S:
            if not self._is_consistent(s, positive_example):
                s_list = list(s)
                for i in range(self.num_features):
                    if s_list[i] == '0':
                        s_list[i] = positive_example[i]
                    elif s_list[i] != positive_example[i]:
                        s_list[i] = '?'
                S_new.append(tuple(s_list))
            else:
                S_new.append(s)
        return list(set(S_new))

    def _remove_inconsistent_G(self, positive_example):
        return [g for g in self.G if self._is_consistent(g, positive_example)]

    def _remove_inconsistent_S(self, negative_example):
        return [s for s in self.S if not self._is_consistent(s, negative_example)]

    def _specialize_G(self, negative_example):
        G_new = []
        for g in self.G:
            if self._is_consistent(g, negative_example):
                for i in range(self.num_features):
                    if g[i] == '?':
                        for value in set(val[i] for val in X if val != negative_example):
                            g_list = list(g)
                            g_list[i] = value
                            if any(self._is_consistent(s, tuple(g_list)) for s in self.S):
                                G_new.append(tuple(g_list))
        return list(set(G_new))

    def _prune_hypotheses(self):
        # Remove redundant hypotheses where one is a generalization of another
        self.S = [h for h in self.S if not any(self._is_more_general(h2, h) for h2 in self.S if h != h2)]
        self.G = [h for h in self.G if not any(self._is_more_general(h, h2) for h2 in self.G if h != h2)]

    def _is_more_general(self, h1, h2):
        """Checks if h1 is more general than or equal to h2."""
        return all(h1[i] == '?' or h1[i] == h2[i] for i in range(self.num_features))

    def get_hypotheses(self):
        return self.S, self.G

# Example usage
X = [
    ('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'),
    ('Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'),
    ('Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'),
    ('Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change')
]
y = [1, 1, 0, 1]  # 1 for positive, 0 for negative

ce = CandidateElimination(num_features=len(X[0]))
ce.fit(X, y)
S_hypotheses, G_hypotheses = ce.get_hypotheses()

print("\nFinal S Hypotheses:")
for s in S_hypotheses:
    print(s)

print("\nFinal G Hypotheses:")
for g in G_hypotheses:
    print(g)


Training example 1: ('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same') with label 1
S after update: [('Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same')]
G after update: [('?', '?', '?', '?', '?', '?')]

Training example 2: ('Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same') with label 1
S after update: [('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')]
G after update: [('?', '?', '?', '?', '?', '?')]

Training example 3: ('Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change') with label 0
S after update: [('Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same')]
G after update: []

Training example 4: ('Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change') with label 1
S after update: [('Sunny', 'Warm', '?', 'Strong', '?', '?')]
G after update: []

Final S Hypotheses:
('Sunny', 'Warm', '?', 'Strong', '?', '?')

Final G Hypotheses:
